# RK1828 VITS 语音合成 Demo


[VITS](https://github.com/jaywalnut310/vits) 端到端英文 TTS。两个预训练模型：

| 模型 | 说话人 |
|---|---|
| LJSpeech | 单一女声 |
| VCTK | 109 个说话人（用 `--speaker_id` 选择） |

每个模型拆成 step1（文本编码 + 时长）和 step2（flow + 声码器）两个 RKNN 模型，
滑窗推理，LJSpeech 实测 RTF 0.345。音素数据依赖 `model/espeak-ng-data/`。

| 步骤 | 内容 |
|---|---|
| 1 | 下载模型（可选 HF / HF-Mirror 源） |
| 2 | 解压预编译 demo |
| 3 | 输入英文文本并合成 |
| 4 | 试听 |

## 0. 环境检查

先确认 RKNN3 运行库（`librknn3_api.so`）和 RK1828 设备都正常。
如失败，请先安装驱动/运行时（`rknn3-rk182x` DEB 包），并用 `sudo rknn-smi info` 确认能看到 RK1828。

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # 加载 /usr/lib/librknn3_api.so；失败说明 Runtime 未安装

devices = find_devices()
if not devices:
    raise RuntimeError("未发现 RK1828 设备！请检查: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"发现设备: id={dev_id}  接口={dev_type}")

## 1. 下载模型

本 demo 的模型文件托管在 Hugging Face（`qhddz` 账号）。
**先用下拉框选择下载源**，再运行下一个单元格开始下载：

- **HF-Mirror 国内镜像**：`hf-mirror.com`，国内直连，推荐
- **Hugging Face 官方**：`huggingface.co`，海外或已有代理时用

> 已存在（且非空）的文件会**自动跳过**，重复运行不会重新下载。

In [ ]:
from nb_download import source_picker

# (HF仓库名, 仓库内路径, 本地路径)
FILES = [
    ("vits-tts-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/vits_tts/demo_x86_64.tar.gz"),
    ("vits-tts-rknn3-rk1828", "model/vits_ljs_step1_slid_fp.rknn", "demos/vits_tts/model/vits_ljs_step1_slid_fp.rknn"),
    ("vits-tts-rknn3-rk1828", "model/vits_ljs_step1_slid_fp.weight", "demos/vits_tts/model/vits_ljs_step1_slid_fp.weight"),
    ("vits-tts-rknn3-rk1828", "model/vits_ljs_step2_slid_fp.rknn", "demos/vits_tts/model/vits_ljs_step2_slid_fp.rknn"),
    ("vits-tts-rknn3-rk1828", "model/vits_ljs_step2_slid_fp.weight", "demos/vits_tts/model/vits_ljs_step2_slid_fp.weight"),
    ("vits-tts-rknn3-rk1828", "model/vits_vctk_step1_slid_fp.rknn", "demos/vits_tts/model/vits_vctk_step1_slid_fp.rknn"),
    ("vits-tts-rknn3-rk1828", "model/vits_vctk_step1_slid_fp.weight", "demos/vits_tts/model/vits_vctk_step1_slid_fp.weight"),
    ("vits-tts-rknn3-rk1828", "model/vits_vctk_step2_slid_fp.rknn", "demos/vits_tts/model/vits_vctk_step2_slid_fp.rknn"),
    ("vits-tts-rknn3-rk1828", "model/vits_vctk_step2_slid_fp.weight", "demos/vits_tts/model/vits_vctk_step2_slid_fp.weight"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/ab_dict", "demos/vits_tts/model/espeak-ng-data/ab_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/af_dict", "demos/vits_tts/model/espeak-ng-data/af_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/am_dict", "demos/vits_tts/model/espeak-ng-data/am_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/an_dict", "demos/vits_tts/model/espeak-ng-data/an_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/ar_dict", "demos/vits_tts/model/espeak-ng-data/ar_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/as_dict", "demos/vits_tts/model/espeak-ng-data/as_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/az_dict", "demos/vits_tts/model/espeak-ng-data/az_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/ba_dict", "demos/vits_tts/model/espeak-ng-data/ba_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/be_dict", "demos/vits_tts/model/espeak-ng-data/be_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/bg_dict", "demos/vits_tts/model/espeak-ng-data/bg_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/bn_dict", "demos/vits_tts/model/espeak-ng-data/bn_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/bpy_dict", "demos/vits_tts/model/espeak-ng-data/bpy_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/bs_dict", "demos/vits_tts/model/espeak-ng-data/bs_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/ca_dict", "demos/vits_tts/model/espeak-ng-data/ca_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/chr_dict", "demos/vits_tts/model/espeak-ng-data/chr_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/cmn_dict", "demos/vits_tts/model/espeak-ng-data/cmn_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/crh_dict", "demos/vits_tts/model/espeak-ng-data/crh_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/cs_dict", "demos/vits_tts/model/espeak-ng-data/cs_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/cv_dict", "demos/vits_tts/model/espeak-ng-data/cv_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/cy_dict", "demos/vits_tts/model/espeak-ng-data/cy_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/da_dict", "demos/vits_tts/model/espeak-ng-data/da_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/de_dict", "demos/vits_tts/model/espeak-ng-data/de_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/el_dict", "demos/vits_tts/model/espeak-ng-data/el_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/en_dict", "demos/vits_tts/model/espeak-ng-data/en_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/eo_dict", "demos/vits_tts/model/espeak-ng-data/eo_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/es_dict", "demos/vits_tts/model/espeak-ng-data/es_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/et_dict", "demos/vits_tts/model/espeak-ng-data/et_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/eu_dict", "demos/vits_tts/model/espeak-ng-data/eu_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/fa_dict", "demos/vits_tts/model/espeak-ng-data/fa_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/fi_dict", "demos/vits_tts/model/espeak-ng-data/fi_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/fr_dict", "demos/vits_tts/model/espeak-ng-data/fr_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/ga_dict", "demos/vits_tts/model/espeak-ng-data/ga_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/gd_dict", "demos/vits_tts/model/espeak-ng-data/gd_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/gn_dict", "demos/vits_tts/model/espeak-ng-data/gn_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/grc_dict", "demos/vits_tts/model/espeak-ng-data/grc_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/gu_dict", "demos/vits_tts/model/espeak-ng-data/gu_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/hak_dict", "demos/vits_tts/model/espeak-ng-data/hak_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/haw_dict", "demos/vits_tts/model/espeak-ng-data/haw_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/he_dict", "demos/vits_tts/model/espeak-ng-data/he_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/hi_dict", "demos/vits_tts/model/espeak-ng-data/hi_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/hr_dict", "demos/vits_tts/model/espeak-ng-data/hr_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/ht_dict", "demos/vits_tts/model/espeak-ng-data/ht_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/hu_dict", "demos/vits_tts/model/espeak-ng-data/hu_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/hy_dict", "demos/vits_tts/model/espeak-ng-data/hy_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/ia_dict", "demos/vits_tts/model/espeak-ng-data/ia_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/id_dict", "demos/vits_tts/model/espeak-ng-data/id_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/intonations", "demos/vits_tts/model/espeak-ng-data/intonations"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/io_dict", "demos/vits_tts/model/espeak-ng-data/io_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/is_dict", "demos/vits_tts/model/espeak-ng-data/is_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/it_dict", "demos/vits_tts/model/espeak-ng-data/it_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/ja_dict", "demos/vits_tts/model/espeak-ng-data/ja_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/jbo_dict", "demos/vits_tts/model/espeak-ng-data/jbo_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/ka_dict", "demos/vits_tts/model/espeak-ng-data/ka_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/kk_dict", "demos/vits_tts/model/espeak-ng-data/kk_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/kl_dict", "demos/vits_tts/model/espeak-ng-data/kl_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/kn_dict", "demos/vits_tts/model/espeak-ng-data/kn_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/ko_dict", "demos/vits_tts/model/espeak-ng-data/ko_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/kok_dict", "demos/vits_tts/model/espeak-ng-data/kok_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/ku_dict", "demos/vits_tts/model/espeak-ng-data/ku_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/ky_dict", "demos/vits_tts/model/espeak-ng-data/ky_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/la_dict", "demos/vits_tts/model/espeak-ng-data/la_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lb_dict", "demos/vits_tts/model/espeak-ng-data/lb_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lfn_dict", "demos/vits_tts/model/espeak-ng-data/lfn_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lt_dict", "demos/vits_tts/model/espeak-ng-data/lt_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lv_dict", "demos/vits_tts/model/espeak-ng-data/lv_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/mi_dict", "demos/vits_tts/model/espeak-ng-data/mi_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/mk_dict", "demos/vits_tts/model/espeak-ng-data/mk_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/ml_dict", "demos/vits_tts/model/espeak-ng-data/ml_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/mn_dict", "demos/vits_tts/model/espeak-ng-data/mn_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/mr_dict", "demos/vits_tts/model/espeak-ng-data/mr_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/ms_dict", "demos/vits_tts/model/espeak-ng-data/ms_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/mt_dict", "demos/vits_tts/model/espeak-ng-data/mt_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/mto_dict", "demos/vits_tts/model/espeak-ng-data/mto_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/my_dict", "demos/vits_tts/model/espeak-ng-data/my_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/nci_dict", "demos/vits_tts/model/espeak-ng-data/nci_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/ne_dict", "demos/vits_tts/model/espeak-ng-data/ne_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/nl_dict", "demos/vits_tts/model/espeak-ng-data/nl_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/no_dict", "demos/vits_tts/model/espeak-ng-data/no_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/nog_dict", "demos/vits_tts/model/espeak-ng-data/nog_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/om_dict", "demos/vits_tts/model/espeak-ng-data/om_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/or_dict", "demos/vits_tts/model/espeak-ng-data/or_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/pa_dict", "demos/vits_tts/model/espeak-ng-data/pa_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/pap_dict", "demos/vits_tts/model/espeak-ng-data/pap_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/phondata", "demos/vits_tts/model/espeak-ng-data/phondata"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/phondata-manifest", "demos/vits_tts/model/espeak-ng-data/phondata-manifest"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/phonindex", "demos/vits_tts/model/espeak-ng-data/phonindex"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/phontab", "demos/vits_tts/model/espeak-ng-data/phontab"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/piqd_dict", "demos/vits_tts/model/espeak-ng-data/piqd_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/pl_dict", "demos/vits_tts/model/espeak-ng-data/pl_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/pt_dict", "demos/vits_tts/model/espeak-ng-data/pt_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/py_dict", "demos/vits_tts/model/espeak-ng-data/py_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/qdb_dict", "demos/vits_tts/model/espeak-ng-data/qdb_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/qu_dict", "demos/vits_tts/model/espeak-ng-data/qu_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/quc_dict", "demos/vits_tts/model/espeak-ng-data/quc_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/qya_dict", "demos/vits_tts/model/espeak-ng-data/qya_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/ro_dict", "demos/vits_tts/model/espeak-ng-data/ro_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/ru_dict", "demos/vits_tts/model/espeak-ng-data/ru_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/rup_dict", "demos/vits_tts/model/espeak-ng-data/rup_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/sd_dict", "demos/vits_tts/model/espeak-ng-data/sd_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/shn_dict", "demos/vits_tts/model/espeak-ng-data/shn_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/si_dict", "demos/vits_tts/model/espeak-ng-data/si_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/sjn_dict", "demos/vits_tts/model/espeak-ng-data/sjn_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/sk_dict", "demos/vits_tts/model/espeak-ng-data/sk_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/sl_dict", "demos/vits_tts/model/espeak-ng-data/sl_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/smj_dict", "demos/vits_tts/model/espeak-ng-data/smj_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/sq_dict", "demos/vits_tts/model/espeak-ng-data/sq_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/sr_dict", "demos/vits_tts/model/espeak-ng-data/sr_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/sv_dict", "demos/vits_tts/model/espeak-ng-data/sv_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/sw_dict", "demos/vits_tts/model/espeak-ng-data/sw_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/ta_dict", "demos/vits_tts/model/espeak-ng-data/ta_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/te_dict", "demos/vits_tts/model/espeak-ng-data/te_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/th_dict", "demos/vits_tts/model/espeak-ng-data/th_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/ti_dict", "demos/vits_tts/model/espeak-ng-data/ti_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/tk_dict", "demos/vits_tts/model/espeak-ng-data/tk_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/tn_dict", "demos/vits_tts/model/espeak-ng-data/tn_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/tr_dict", "demos/vits_tts/model/espeak-ng-data/tr_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/tt_dict", "demos/vits_tts/model/espeak-ng-data/tt_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/ug_dict", "demos/vits_tts/model/espeak-ng-data/ug_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/uk_dict", "demos/vits_tts/model/espeak-ng-data/uk_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/ur_dict", "demos/vits_tts/model/espeak-ng-data/ur_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/uz_dict", "demos/vits_tts/model/espeak-ng-data/uz_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/vi_dict", "demos/vits_tts/model/espeak-ng-data/vi_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/yue_dict", "demos/vits_tts/model/espeak-ng-data/yue_dict"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Alex", "demos/vits_tts/model/espeak-ng-data/voices/!v/Alex"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Alicia", "demos/vits_tts/model/espeak-ng-data/voices/!v/Alicia"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Andrea", "demos/vits_tts/model/espeak-ng-data/voices/!v/Andrea"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Andy", "demos/vits_tts/model/espeak-ng-data/voices/!v/Andy"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Annie", "demos/vits_tts/model/espeak-ng-data/voices/!v/Annie"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/AnxiousAndy", "demos/vits_tts/model/espeak-ng-data/voices/!v/AnxiousAndy"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Demonic", "demos/vits_tts/model/espeak-ng-data/voices/!v/Demonic"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Denis", "demos/vits_tts/model/espeak-ng-data/voices/!v/Denis"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Diogo", "demos/vits_tts/model/espeak-ng-data/voices/!v/Diogo"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Gene", "demos/vits_tts/model/espeak-ng-data/voices/!v/Gene"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Gene2", "demos/vits_tts/model/espeak-ng-data/voices/!v/Gene2"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Henrique", "demos/vits_tts/model/espeak-ng-data/voices/!v/Henrique"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Hugo", "demos/vits_tts/model/espeak-ng-data/voices/!v/Hugo"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Jacky", "demos/vits_tts/model/espeak-ng-data/voices/!v/Jacky"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Lee", "demos/vits_tts/model/espeak-ng-data/voices/!v/Lee"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Marco", "demos/vits_tts/model/espeak-ng-data/voices/!v/Marco"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Mario", "demos/vits_tts/model/espeak-ng-data/voices/!v/Mario"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Michael", "demos/vits_tts/model/espeak-ng-data/voices/!v/Michael"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Mike", "demos/vits_tts/model/espeak-ng-data/voices/!v/Mike"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Mr serious", "demos/vits_tts/model/espeak-ng-data/voices/!v/Mr serious"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Nguyen", "demos/vits_tts/model/espeak-ng-data/voices/!v/Nguyen"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Reed", "demos/vits_tts/model/espeak-ng-data/voices/!v/Reed"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/RicishayMax", "demos/vits_tts/model/espeak-ng-data/voices/!v/RicishayMax"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/RicishayMax2", "demos/vits_tts/model/espeak-ng-data/voices/!v/RicishayMax2"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/RicishayMax3", "demos/vits_tts/model/espeak-ng-data/voices/!v/RicishayMax3"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Storm", "demos/vits_tts/model/espeak-ng-data/voices/!v/Storm"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/Tweaky", "demos/vits_tts/model/espeak-ng-data/voices/!v/Tweaky"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/UniRobot", "demos/vits_tts/model/espeak-ng-data/voices/!v/UniRobot"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/adam", "demos/vits_tts/model/espeak-ng-data/voices/!v/adam"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/anika", "demos/vits_tts/model/espeak-ng-data/voices/!v/anika"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/anikaRobot", "demos/vits_tts/model/espeak-ng-data/voices/!v/anikaRobot"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/announcer", "demos/vits_tts/model/espeak-ng-data/voices/!v/announcer"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/antonio", "demos/vits_tts/model/espeak-ng-data/voices/!v/antonio"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/aunty", "demos/vits_tts/model/espeak-ng-data/voices/!v/aunty"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/belinda", "demos/vits_tts/model/espeak-ng-data/voices/!v/belinda"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/benjamin", "demos/vits_tts/model/espeak-ng-data/voices/!v/benjamin"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/boris", "demos/vits_tts/model/espeak-ng-data/voices/!v/boris"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/caleb", "demos/vits_tts/model/espeak-ng-data/voices/!v/caleb"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/croak", "demos/vits_tts/model/espeak-ng-data/voices/!v/croak"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/david", "demos/vits_tts/model/espeak-ng-data/voices/!v/david"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/ed", "demos/vits_tts/model/espeak-ng-data/voices/!v/ed"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/edward", "demos/vits_tts/model/espeak-ng-data/voices/!v/edward"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/edward2", "demos/vits_tts/model/espeak-ng-data/voices/!v/edward2"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/f1", "demos/vits_tts/model/espeak-ng-data/voices/!v/f1"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/f2", "demos/vits_tts/model/espeak-ng-data/voices/!v/f2"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/f3", "demos/vits_tts/model/espeak-ng-data/voices/!v/f3"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/f4", "demos/vits_tts/model/espeak-ng-data/voices/!v/f4"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/f5", "demos/vits_tts/model/espeak-ng-data/voices/!v/f5"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/fast", "demos/vits_tts/model/espeak-ng-data/voices/!v/fast"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/grandma", "demos/vits_tts/model/espeak-ng-data/voices/!v/grandma"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/grandpa", "demos/vits_tts/model/espeak-ng-data/voices/!v/grandpa"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/gustave", "demos/vits_tts/model/espeak-ng-data/voices/!v/gustave"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/ian", "demos/vits_tts/model/espeak-ng-data/voices/!v/ian"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/iven", "demos/vits_tts/model/espeak-ng-data/voices/!v/iven"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/iven2", "demos/vits_tts/model/espeak-ng-data/voices/!v/iven2"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/iven3", "demos/vits_tts/model/espeak-ng-data/voices/!v/iven3"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/iven4", "demos/vits_tts/model/espeak-ng-data/voices/!v/iven4"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/john", "demos/vits_tts/model/espeak-ng-data/voices/!v/john"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/kaukovalta", "demos/vits_tts/model/espeak-ng-data/voices/!v/kaukovalta"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/klatt", "demos/vits_tts/model/espeak-ng-data/voices/!v/klatt"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/klatt2", "demos/vits_tts/model/espeak-ng-data/voices/!v/klatt2"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/klatt3", "demos/vits_tts/model/espeak-ng-data/voices/!v/klatt3"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/klatt4", "demos/vits_tts/model/espeak-ng-data/voices/!v/klatt4"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/klatt5", "demos/vits_tts/model/espeak-ng-data/voices/!v/klatt5"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/klatt6", "demos/vits_tts/model/espeak-ng-data/voices/!v/klatt6"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/linda", "demos/vits_tts/model/espeak-ng-data/voices/!v/linda"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/m1", "demos/vits_tts/model/espeak-ng-data/voices/!v/m1"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/m2", "demos/vits_tts/model/espeak-ng-data/voices/!v/m2"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/m3", "demos/vits_tts/model/espeak-ng-data/voices/!v/m3"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/m4", "demos/vits_tts/model/espeak-ng-data/voices/!v/m4"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/m5", "demos/vits_tts/model/espeak-ng-data/voices/!v/m5"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/m6", "demos/vits_tts/model/espeak-ng-data/voices/!v/m6"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/m7", "demos/vits_tts/model/espeak-ng-data/voices/!v/m7"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/m8", "demos/vits_tts/model/espeak-ng-data/voices/!v/m8"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/marcelo", "demos/vits_tts/model/espeak-ng-data/voices/!v/marcelo"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/max", "demos/vits_tts/model/espeak-ng-data/voices/!v/max"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/michel", "demos/vits_tts/model/espeak-ng-data/voices/!v/michel"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/miguel", "demos/vits_tts/model/espeak-ng-data/voices/!v/miguel"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/mike2", "demos/vits_tts/model/espeak-ng-data/voices/!v/mike2"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/norbert", "demos/vits_tts/model/espeak-ng-data/voices/!v/norbert"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/pablo", "demos/vits_tts/model/espeak-ng-data/voices/!v/pablo"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/paul", "demos/vits_tts/model/espeak-ng-data/voices/!v/paul"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/pedro", "demos/vits_tts/model/espeak-ng-data/voices/!v/pedro"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/quincy", "demos/vits_tts/model/espeak-ng-data/voices/!v/quincy"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/rob", "demos/vits_tts/model/espeak-ng-data/voices/!v/rob"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/robert", "demos/vits_tts/model/espeak-ng-data/voices/!v/robert"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/robosoft", "demos/vits_tts/model/espeak-ng-data/voices/!v/robosoft"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/robosoft2", "demos/vits_tts/model/espeak-ng-data/voices/!v/robosoft2"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/robosoft3", "demos/vits_tts/model/espeak-ng-data/voices/!v/robosoft3"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/robosoft4", "demos/vits_tts/model/espeak-ng-data/voices/!v/robosoft4"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/robosoft5", "demos/vits_tts/model/espeak-ng-data/voices/!v/robosoft5"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/robosoft6", "demos/vits_tts/model/espeak-ng-data/voices/!v/robosoft6"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/robosoft7", "demos/vits_tts/model/espeak-ng-data/voices/!v/robosoft7"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/robosoft8", "demos/vits_tts/model/espeak-ng-data/voices/!v/robosoft8"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/sandro", "demos/vits_tts/model/espeak-ng-data/voices/!v/sandro"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/shelby", "demos/vits_tts/model/espeak-ng-data/voices/!v/shelby"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/steph", "demos/vits_tts/model/espeak-ng-data/voices/!v/steph"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/steph2", "demos/vits_tts/model/espeak-ng-data/voices/!v/steph2"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/steph3", "demos/vits_tts/model/espeak-ng-data/voices/!v/steph3"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/travis", "demos/vits_tts/model/espeak-ng-data/voices/!v/travis"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/victor", "demos/vits_tts/model/espeak-ng-data/voices/!v/victor"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/whisper", "demos/vits_tts/model/espeak-ng-data/voices/!v/whisper"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/whisperf", "demos/vits_tts/model/espeak-ng-data/voices/!v/whisperf"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/voices/!v/zac", "demos/vits_tts/model/espeak-ng-data/voices/!v/zac"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/eu", "demos/vits_tts/model/espeak-ng-data/lang/eu"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/ko", "demos/vits_tts/model/espeak-ng-data/lang/ko"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/qu", "demos/vits_tts/model/espeak-ng-data/lang/qu"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/ira/fa", "demos/vits_tts/model/espeak-ng-data/lang/ira/fa"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/ira/fa-Latn", "demos/vits_tts/model/espeak-ng-data/lang/ira/fa-Latn"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/ira/ku", "demos/vits_tts/model/espeak-ng-data/lang/ira/ku"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/ira/ps", "demos/vits_tts/model/espeak-ng-data/lang/ira/ps"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/map/haw", "demos/vits_tts/model/espeak-ng-data/lang/map/haw"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/art/eo", "demos/vits_tts/model/espeak-ng-data/lang/art/eo"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/art/ia", "demos/vits_tts/model/espeak-ng-data/lang/art/ia"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/art/io", "demos/vits_tts/model/espeak-ng-data/lang/art/io"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/art/jbo", "demos/vits_tts/model/espeak-ng-data/lang/art/jbo"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/art/lfn", "demos/vits_tts/model/espeak-ng-data/lang/art/lfn"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/art/piqd", "demos/vits_tts/model/espeak-ng-data/lang/art/piqd"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/art/py", "demos/vits_tts/model/espeak-ng-data/lang/art/py"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/art/qdb", "demos/vits_tts/model/espeak-ng-data/lang/art/qdb"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/art/qya", "demos/vits_tts/model/espeak-ng-data/lang/art/qya"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/art/sjn", "demos/vits_tts/model/espeak-ng-data/lang/art/sjn"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/art/xex", "demos/vits_tts/model/espeak-ng-data/lang/art/xex"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/dra/kn", "demos/vits_tts/model/espeak-ng-data/lang/dra/kn"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/dra/ml", "demos/vits_tts/model/espeak-ng-data/lang/dra/ml"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/dra/ta", "demos/vits_tts/model/espeak-ng-data/lang/dra/ta"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/dra/te", "demos/vits_tts/model/espeak-ng-data/lang/dra/te"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/zlw/cs", "demos/vits_tts/model/espeak-ng-data/lang/zlw/cs"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/zlw/pl", "demos/vits_tts/model/espeak-ng-data/lang/zlw/pl"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/zlw/sk", "demos/vits_tts/model/espeak-ng-data/lang/zlw/sk"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/cau/ab", "demos/vits_tts/model/espeak-ng-data/lang/cau/ab"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/ccs/ka", "demos/vits_tts/model/espeak-ng-data/lang/ccs/ka"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/trk/az", "demos/vits_tts/model/espeak-ng-data/lang/trk/az"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/trk/ba", "demos/vits_tts/model/espeak-ng-data/lang/trk/ba"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/trk/crh", "demos/vits_tts/model/espeak-ng-data/lang/trk/crh"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/trk/cv", "demos/vits_tts/model/espeak-ng-data/lang/trk/cv"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/trk/kaa", "demos/vits_tts/model/espeak-ng-data/lang/trk/kaa"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/trk/kk", "demos/vits_tts/model/espeak-ng-data/lang/trk/kk"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/trk/ky", "demos/vits_tts/model/espeak-ng-data/lang/trk/ky"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/trk/nog", "demos/vits_tts/model/espeak-ng-data/lang/trk/nog"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/trk/tk", "demos/vits_tts/model/espeak-ng-data/lang/trk/tk"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/trk/tr", "demos/vits_tts/model/espeak-ng-data/lang/trk/tr"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/trk/tt", "demos/vits_tts/model/espeak-ng-data/lang/trk/tt"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/trk/ug", "demos/vits_tts/model/espeak-ng-data/lang/trk/ug"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/trk/uz", "demos/vits_tts/model/espeak-ng-data/lang/trk/uz"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/miz/mto", "demos/vits_tts/model/espeak-ng-data/lang/miz/mto"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/zls/bg", "demos/vits_tts/model/espeak-ng-data/lang/zls/bg"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/zls/bs", "demos/vits_tts/model/espeak-ng-data/lang/zls/bs"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/zls/hr", "demos/vits_tts/model/espeak-ng-data/lang/zls/hr"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/zls/mk", "demos/vits_tts/model/espeak-ng-data/lang/zls/mk"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/zls/sl", "demos/vits_tts/model/espeak-ng-data/lang/zls/sl"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/zls/sr", "demos/vits_tts/model/espeak-ng-data/lang/zls/sr"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/sai/gn", "demos/vits_tts/model/espeak-ng-data/lang/sai/gn"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/esx/kl", "demos/vits_tts/model/espeak-ng-data/lang/esx/kl"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/myn/quc", "demos/vits_tts/model/espeak-ng-data/lang/myn/quc"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/bat/lt", "demos/vits_tts/model/espeak-ng-data/lang/bat/lt"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/bat/ltg", "demos/vits_tts/model/espeak-ng-data/lang/bat/ltg"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/bat/lv", "demos/vits_tts/model/espeak-ng-data/lang/bat/lv"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/xgn/mn", "demos/vits_tts/model/espeak-ng-data/lang/xgn/mn"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/jpx/ja", "demos/vits_tts/model/espeak-ng-data/lang/jpx/ja"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/poz/id", "demos/vits_tts/model/espeak-ng-data/lang/poz/id"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/poz/mi", "demos/vits_tts/model/espeak-ng-data/lang/poz/mi"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/poz/ms", "demos/vits_tts/model/espeak-ng-data/lang/poz/ms"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/sem/am", "demos/vits_tts/model/espeak-ng-data/lang/sem/am"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/sem/ar", "demos/vits_tts/model/espeak-ng-data/lang/sem/ar"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/sem/he", "demos/vits_tts/model/espeak-ng-data/lang/sem/he"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/sem/mt", "demos/vits_tts/model/espeak-ng-data/lang/sem/mt"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/sem/ti", "demos/vits_tts/model/espeak-ng-data/lang/sem/ti"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/inc/as", "demos/vits_tts/model/espeak-ng-data/lang/inc/as"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/inc/bn", "demos/vits_tts/model/espeak-ng-data/lang/inc/bn"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/inc/bpy", "demos/vits_tts/model/espeak-ng-data/lang/inc/bpy"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/inc/gu", "demos/vits_tts/model/espeak-ng-data/lang/inc/gu"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/inc/hi", "demos/vits_tts/model/espeak-ng-data/lang/inc/hi"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/inc/kok", "demos/vits_tts/model/espeak-ng-data/lang/inc/kok"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/inc/mr", "demos/vits_tts/model/espeak-ng-data/lang/inc/mr"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/inc/ne", "demos/vits_tts/model/espeak-ng-data/lang/inc/ne"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/inc/or", "demos/vits_tts/model/espeak-ng-data/lang/inc/or"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/inc/pa", "demos/vits_tts/model/espeak-ng-data/lang/inc/pa"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/inc/sd", "demos/vits_tts/model/espeak-ng-data/lang/inc/sd"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/inc/si", "demos/vits_tts/model/espeak-ng-data/lang/inc/si"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/inc/ur", "demos/vits_tts/model/espeak-ng-data/lang/inc/ur"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/tai/shn", "demos/vits_tts/model/espeak-ng-data/lang/tai/shn"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/tai/th", "demos/vits_tts/model/espeak-ng-data/lang/tai/th"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/grk/el", "demos/vits_tts/model/espeak-ng-data/lang/grk/el"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/grk/grc", "demos/vits_tts/model/espeak-ng-data/lang/grk/grc"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/cus/om", "demos/vits_tts/model/espeak-ng-data/lang/cus/om"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/zle/be", "demos/vits_tts/model/espeak-ng-data/lang/zle/be"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/zle/ru", "demos/vits_tts/model/espeak-ng-data/lang/zle/ru"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/zle/ru-LV", "demos/vits_tts/model/espeak-ng-data/lang/zle/ru-LV"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/zle/ru-cl", "demos/vits_tts/model/espeak-ng-data/lang/zle/ru-cl"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/zle/uk", "demos/vits_tts/model/espeak-ng-data/lang/zle/uk"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/aav/vi", "demos/vits_tts/model/espeak-ng-data/lang/aav/vi"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/aav/vi-VN-x-central", "demos/vits_tts/model/espeak-ng-data/lang/aav/vi-VN-x-central"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/aav/vi-VN-x-south", "demos/vits_tts/model/espeak-ng-data/lang/aav/vi-VN-x-south"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/bnt/sw", "demos/vits_tts/model/espeak-ng-data/lang/bnt/sw"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/bnt/tn", "demos/vits_tts/model/espeak-ng-data/lang/bnt/tn"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/azc/nci", "demos/vits_tts/model/espeak-ng-data/lang/azc/nci"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/iro/chr", "demos/vits_tts/model/espeak-ng-data/lang/iro/chr"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/itc/la", "demos/vits_tts/model/espeak-ng-data/lang/itc/la"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/gmw/af", "demos/vits_tts/model/espeak-ng-data/lang/gmw/af"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/gmw/de", "demos/vits_tts/model/espeak-ng-data/lang/gmw/de"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/gmw/en", "demos/vits_tts/model/espeak-ng-data/lang/gmw/en"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/gmw/en-029", "demos/vits_tts/model/espeak-ng-data/lang/gmw/en-029"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/gmw/en-GB-scotland", "demos/vits_tts/model/espeak-ng-data/lang/gmw/en-GB-scotland"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/gmw/en-GB-x-gbclan", "demos/vits_tts/model/espeak-ng-data/lang/gmw/en-GB-x-gbclan"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/gmw/en-GB-x-gbcwmd", "demos/vits_tts/model/espeak-ng-data/lang/gmw/en-GB-x-gbcwmd"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/gmw/en-GB-x-rp", "demos/vits_tts/model/espeak-ng-data/lang/gmw/en-GB-x-rp"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/gmw/en-Shaw", "demos/vits_tts/model/espeak-ng-data/lang/gmw/en-Shaw"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/gmw/en-US", "demos/vits_tts/model/espeak-ng-data/lang/gmw/en-US"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/gmw/en-US-nyc", "demos/vits_tts/model/espeak-ng-data/lang/gmw/en-US-nyc"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/gmw/lb", "demos/vits_tts/model/espeak-ng-data/lang/gmw/lb"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/gmw/nl", "demos/vits_tts/model/espeak-ng-data/lang/gmw/nl"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/ine/hy", "demos/vits_tts/model/espeak-ng-data/lang/ine/hy"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/ine/hyw", "demos/vits_tts/model/espeak-ng-data/lang/ine/hyw"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/ine/sq", "demos/vits_tts/model/espeak-ng-data/lang/ine/sq"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/cel/cy", "demos/vits_tts/model/espeak-ng-data/lang/cel/cy"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/cel/ga", "demos/vits_tts/model/espeak-ng-data/lang/cel/ga"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/cel/gd", "demos/vits_tts/model/espeak-ng-data/lang/cel/gd"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/sit/cmn", "demos/vits_tts/model/espeak-ng-data/lang/sit/cmn"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/sit/cmn-Latn-pinyin", "demos/vits_tts/model/espeak-ng-data/lang/sit/cmn-Latn-pinyin"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/sit/hak", "demos/vits_tts/model/espeak-ng-data/lang/sit/hak"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/sit/my", "demos/vits_tts/model/espeak-ng-data/lang/sit/my"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/sit/yue", "demos/vits_tts/model/espeak-ng-data/lang/sit/yue"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/sit/yue-Latn-jyutping", "demos/vits_tts/model/espeak-ng-data/lang/sit/yue-Latn-jyutping"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/urj/et", "demos/vits_tts/model/espeak-ng-data/lang/urj/et"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/urj/fi", "demos/vits_tts/model/espeak-ng-data/lang/urj/fi"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/urj/hu", "demos/vits_tts/model/espeak-ng-data/lang/urj/hu"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/urj/smj", "demos/vits_tts/model/espeak-ng-data/lang/urj/smj"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/roa/an", "demos/vits_tts/model/espeak-ng-data/lang/roa/an"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/roa/ca", "demos/vits_tts/model/espeak-ng-data/lang/roa/ca"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/roa/ca-ba", "demos/vits_tts/model/espeak-ng-data/lang/roa/ca-ba"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/roa/ca-nw", "demos/vits_tts/model/espeak-ng-data/lang/roa/ca-nw"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/roa/ca-va", "demos/vits_tts/model/espeak-ng-data/lang/roa/ca-va"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/roa/es", "demos/vits_tts/model/espeak-ng-data/lang/roa/es"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/roa/es-419", "demos/vits_tts/model/espeak-ng-data/lang/roa/es-419"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/roa/fr", "demos/vits_tts/model/espeak-ng-data/lang/roa/fr"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/roa/fr-BE", "demos/vits_tts/model/espeak-ng-data/lang/roa/fr-BE"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/roa/fr-CH", "demos/vits_tts/model/espeak-ng-data/lang/roa/fr-CH"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/roa/ht", "demos/vits_tts/model/espeak-ng-data/lang/roa/ht"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/roa/it", "demos/vits_tts/model/espeak-ng-data/lang/roa/it"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/roa/pap", "demos/vits_tts/model/espeak-ng-data/lang/roa/pap"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/roa/pt", "demos/vits_tts/model/espeak-ng-data/lang/roa/pt"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/roa/pt-BR", "demos/vits_tts/model/espeak-ng-data/lang/roa/pt-BR"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/roa/ro", "demos/vits_tts/model/espeak-ng-data/lang/roa/ro"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/roa/rup", "demos/vits_tts/model/espeak-ng-data/lang/roa/rup"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/gmq/da", "demos/vits_tts/model/espeak-ng-data/lang/gmq/da"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/gmq/fo", "demos/vits_tts/model/espeak-ng-data/lang/gmq/fo"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/gmq/is", "demos/vits_tts/model/espeak-ng-data/lang/gmq/is"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/gmq/nb", "demos/vits_tts/model/espeak-ng-data/lang/gmq/nb"),
    ("vits-tts-rknn3-rk1828", "model/espeak-ng-data/lang/gmq/sv", "demos/vits_tts/model/espeak-ng-data/lang/gmq/sv"),
]

w_source = source_picker()

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, w_source.value)

## 2. 解压预编译 demo

`demo_x86_64.tar.gz` 里是官方流程编译好的 x86_64 C++ demo 和运行时库（`lib/`）。
HF 不允许直接托管可执行文件，所以打包成 tar.gz，解压一次即可（重复运行自动跳过）。

> **为什么这个 demo 用 C++ 而不是纯 Python？**
> 01-05（分类/检测/FastSAM/pose/分割）是单个模型一次前向，后处理简单，
> 所以用 Python（`rknn3.py`）逐步演示。而本 demo 的流水线复杂得多——
> 多模型串联、tokenizer、KV cache、滑窗/流式状态等，前后处理和编排逻辑
> 有几百上千行，重写 Python 版成本高且容易和官方实现不一致。
> 因此直接运行官方 `rknn3-model-zoo` 编译的 C++ demo，
> 源码见 [github.com/airockchip/rknn3-model-zoo](https://github.com/airockchip/rknn3-model-zoo)
> 的 `examples/` 目录。

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/vits_tts/demo_x86_64.tar.gz", "demos/vits_tts")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. 输入文本并合成

In [ ]:
import ipywidgets as widgets
from IPython.display import display

w_model = widgets.Dropdown(options=[("LJSpeech 单女声", "ljs"),
                                    ("VCTK 多说话人", "vctk")],
                           description="模型:")
w_speaker = widgets.IntSlider(value=4, min=0, max=108,
                              description="说话人ID:",
                              style={"description_width": "80px"})
w_text = widgets.Textarea(value="VITS is awesome! Have a nice day.",
                          description="文本:",
                          layout=widgets.Layout(width="70%", height="50px"))
display(w_model, w_speaker, w_text)
print("选好后继续往下运行 ▼（说话人ID 仅 VCTK 有效）")

In [ ]:
DEMO = "demos/vits_tts"
before = snapshot(DEMO)
args = ["./rknn_vits_demo",
        "--step1_model", f"model/vits_{w_model.value}_step1_slid_fp.rknn",
        "--step1_weight", f"model/vits_{w_model.value}_step1_slid_fp.weight",
        "--step2_model", f"model/vits_{w_model.value}_step2_slid_fp.rknn",
        "--step2_weight", f"model/vits_{w_model.value}_step2_slid_fp.weight",
        "--text", w_text.value,
        "--output", "out.wav"]
if w_model.value == "vctk":
    args += ["--speaker_id", str(w_speaker.value)]
run_demo(args, DEMO)

## 4. 试听合成结果

In [ ]:
show_audio("demos/vits_tts/out.wav")

## 小结

- TTS 流水线：文本 -> 音素（espeak-ng）-> step1 出时长/声学特征 -> step2 出声波。
- 换几个 speaker_id 对比不同说话人的音色。